# Hawker Centre Distance & CBD Distance Pipeline

This notebook computes amenity distances for each HDB resale flat address:
1. Distance to the nearest hawker centre **open at the time of the transaction**
2. Straight-line distance to the CBD (Raffles Place MRT proxy)
3. Distance to the nearest MOE primary school

All distances are computed from cached coordinates — no API calls are needed after the first run.

## Inputs
- `../merged_data/hdb_with_mrt_distances.csv` — enriched HDB dataset (already contains `lat`, `lon`)
- `../data/geocode_cache.json` — geocoded lat/lon per address (reference)
- `../data/HawkerCentresGEOJSON.geojson` — hawker centre locations and status metadata
- `../data/general-information-of-schools.csv` — MOE school directory (Phase 6)

## How to Run
Run cells top to bottom. Phases 3 and 5 are fully vectorised and run in seconds. Phase 6 geocodes primary schools via the OneMap API on first run (~60 s for ~186 schools); subsequent runs use the cache and complete instantly.

## Output Files
| File | Description |
|------|-------------|
| `../merged_data/hdb_with_amenities_macro.csv` | Full dataset enriched with `dist_nearest_hawker_m`, `dist_cbd_m`, and `dist_nearest_primary_m` |

## Hawker Centre Status Rules
| STATUS | Treatment |
|--------|-----------|
| `Existing` | Always open (all predate 2017; dates are year-only strings) |
| `Existing (replacement)` | Always open (location had a hawker centre before the rebuild) |
| `Existing (new)` | Open only if `EST_ORIGINAL_COMPLETION_DATE ≤ transaction month` |
| `Interim Centre` | Open only if `EST_ORIGINAL_COMPLETION_DATE ≤ transaction month` |
| `Under Construction` | Always excluded |

In [7]:
import pandas as pd
import numpy as np
import json
import os

print("Libraries imported successfully!")

Libraries imported successfully!


## Phase 1 — Load Data

Load the enriched HDB dataset (which already contains geocoded `lat`/`lon` from `mrt_pipeline.ipynb`) and the geocode cache for reference.

In [8]:
GEOCODE_CACHE_FILE = '../data/geocode_cache.json'
INPUT_CSV          = '../merged_data/hdb_with_mrt_distances.csv'

with open(GEOCODE_CACHE_FILE) as f:
    geocode_cache = json.load(f)
print(f"Geocode cache: {len(geocode_cache):,} addresses")

df = pd.read_csv(INPUT_CSV)
df['month'] = pd.to_datetime(df['month'])
print(f"Dataset shape: {df.shape}")
print(f"Date range:    {df['month'].min().strftime('%Y-%m')} to {df['month'].max().strftime('%Y-%m')}")
print(f"Rows with lat/lon: {df['lat'].notna().sum():,}")
print()
print(df[['month', 'block', 'street_name', 'lat', 'lon']].head(5))

Geocode cache: 9,660 addresses
Dataset shape: (197432, 23)
Date range:    2018-04 to 2025-12
Rows with lat/lon: 197,327

       month block        street_name       lat         lon
0 2018-04-01   314   ANG MO KIO AVE 3  1.366227  103.850086
1 2018-04-01   314   ANG MO KIO AVE 3  1.366227  103.850086
2 2018-04-01   156   ANG MO KIO AVE 4  1.375495  103.839947
3 2018-04-01   463  ANG MO KIO AVE 10  1.366948  103.857488
4 2018-04-01   503   ANG MO KIO AVE 5  1.375462  103.848942


## Phase 2 — Load Hawker Centre Data

Parse the GeoJSON into a flat DataFrame. For `Existing (new)` and `Interim Centre` entries, parse `EST_ORIGINAL_COMPLETION_DATE` with `dayfirst=True` to handle the `DD/M/YYYY` format. Parse failures (e.g. year-only strings like `"1983"` for `Existing` entries) produce `NaT` and are treated as always-open. Pre-compute the set of open hawker centres for every unique transaction month in the dataset.

In [9]:
HAWKER_GEOJSON = '../data/HawkerCentresGEOJSON.geojson'

with open(HAWKER_GEOJSON) as f:
    geojson = json.load(f)

rows = []
for feat in geojson['features']:
    props = feat['properties']
    lon, lat = feat['geometry']['coordinates']   # GeoJSON standard: [lon, lat]
    date_str = props.get('EST_ORIGINAL_COMPLETION_DATE', '')
    est_dt   = pd.to_datetime(date_str, dayfirst=True, errors='coerce')
    rows.append({
        'name':              props.get('NAME', ''),
        'status':            props.get('STATUS', ''),
        'est_completion_dt': est_dt,
        'lat': lat,
        'lon': lon,
    })
hawker_df = pd.DataFrame(rows)

print(f"Total hawker centres loaded: {len(hawker_df):,}")
print("\nBy STATUS:")
print(hawker_df['status'].value_counts().to_string())

date_dep = hawker_df.query("status in ['Existing (new)', 'Interim Centre']").copy()
print(f"\nDate-dependent centres ({len(date_dep)}) — filtered by EST_ORIGINAL_COMPLETION_DATE:")
print(date_dep[['name', 'status', 'est_completion_dt']].to_string())

Total hawker centres loaded: 129

By STATUS:
status
Existing                  103
Existing (new)             16
Under Construction          6
Existing (replacement)      3
Interim Centre              1

Date-dependent centres (17) — filtered by EST_ORIGINAL_COMPLETION_DATE:
                                             name          status est_completion_dt
2                     Punggol Coast Hawker Centre  Existing (new)        2024-08-21
5    Bukit Timah Interim Hawker Centre and Market  Interim Centre        2024-10-01
11                      Jurong West Hawker Centre  Existing (new)        2017-07-28
15                      One Punggol Hawker Centre  Existing (new)        2022-04-29
17                   Bukit Canberra Hawker Centre  Existing (new)        2022-09-27
23                      Yishun Park Hawker Centre  Existing (new)        2017-07-03
32                 Bukit Batok West Hawker Centre  Existing (new)        2024-10-08
33                Fernvale Hawker Centre & Market  Ex

In [10]:
def is_hawker_open(status, est_dt, transaction_month):
    """Return True if the hawker centre was open at transaction_month."""
    if status == 'Under Construction':
        return False
    if status in ('Existing', 'Existing (replacement)'):
        return True          # always open; predates dataset or rebuilt on same site
    # 'Existing (new)' or 'Interim Centre': date-dependent
    if pd.isna(est_dt):
        return True          # parse failure → safe fallback (treat as always open)
    return transaction_month >= est_dt


def hawker_dists_vectorized(lats_arr, lons_arr, hawker_coords):
    """Compute minimum haversine distance (metres) from N addresses to H hawker centres.

    Parameters
    ----------
    lats_arr, lons_arr : 1-D float arrays, shape (N,)
    hawker_coords      : 2-D float array, shape (H, 2), columns [lat, lon]

    Returns
    -------
    1-D float array, shape (N,) — NaN for rows where lat/lon is NaN.
    """
    R = 6_371_000.0
    if len(hawker_coords) == 0:
        return np.full(len(lats_arr), np.nan)

    valid  = ~(np.isnan(lats_arr) | np.isnan(lons_arr))
    result = np.full(len(lats_arr), np.nan)
    if not valid.any():
        return result

    lat1 = np.radians(lats_arr[valid])[:, np.newaxis]       # (N_v, 1)
    lon1 = np.radians(lons_arr[valid])[:, np.newaxis]
    lat2 = np.radians(hawker_coords[:, 0])[np.newaxis, :]   # (1, H)
    lon2 = np.radians(hawker_coords[:, 1])[np.newaxis, :]

    a      = (np.sin((lat2 - lat1) / 2) ** 2
               + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    dist_m = R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))  # (N_v, H)
    result[valid] = np.round(dist_m.min(axis=1), 1)
    return result


# Pre-compute open-hawker coordinate arrays for every unique month in the dataset.
# Stored as a dict: month (Timestamp) → numpy array of shape (H_open, 2).
print("Pre-computing open hawker sets per unique month...")
unique_months = sorted(df['month'].unique())
open_hawkers_by_month = {}
for month in unique_months:
    mask = hawker_df.apply(
        lambda r: is_hawker_open(r['status'], r['est_completion_dt'], month), axis=1)
    open_hawkers_by_month[month] = hawker_df.loc[mask, ['lat', 'lon']].values

# Spot-check: open counts at key dates
for m_str in ['2018-04-01', '2021-01-01', '2024-07-01']:
    m = pd.Timestamp(m_str)
    if m in open_hawkers_by_month:
        print(f"  {m_str[:7]}: {len(open_hawkers_by_month[m]):>3} hawker centres open")
print(f"Pre-computation complete for {len(unique_months)} unique months.")

Pre-computing open hawker sets per unique month...
  2018-04: 113 hawker centres open
  2021-01: 113 hawker centres open
  2024-07: 120 hawker centres open
Pre-computation complete for 93 unique months.


## Phase 3 — Compute Nearest Hawker Distance

For each transaction row, look up the set of hawker centres open at that transaction month, then compute the haversine distance to all of them and take the minimum. The computation is fully vectorised using NumPy broadcasting — no API calls are made. All rows sharing the same month are processed together in a single matrix operation.

### [ SAMPLE TEST ] — Verify distance logic on a small subset
Run this first to confirm the output looks sensible before processing the full dataset.

In [11]:
# Sample: mix of towns + include Punggol rows (near hawker centres that opened later)
# to verify that temporal filtering works correctly.
sample_rows = pd.concat([
    df[df['street_name'].str.contains('PUNGGOL', na=False)].head(4),
    df[df['town'] == 'BISHAN'].head(3),
    df[df['town'] == 'JURONG WEST'].head(3),
]).dropna(subset=['lat', 'lon']).head(10)

print(f"Sample test: {len(sample_rows)} rows\n")
print(f"{'Month':<8}  {'Block':<6}  {'Street':<32}  {'Open HCs':>8}  {'Dist (m)':>10}")
print('-' * 74)
for _, row in sample_rows.iterrows():
    coords   = open_hawkers_by_month.get(row['month'], np.zeros((0, 2)))
    dist     = hawker_dists_vectorized(
                   np.array([row['lat']]), np.array([row['lon']]), coords)[0]
    dist_str = f"{dist:.1f}" if not np.isnan(dist) else "NaN"
    print(f"{str(row['month'])[:7]:<8}  {str(row['block']):<6}  "
          f"{row['street_name']:<32}  {len(coords):>8}  {dist_str:>10}")

Sample test: 10 rows

Month     Block   Street                            Open HCs    Dist (m)
--------------------------------------------------------------------------
2018-04   622C    PUNGGOL CTRL                           113      4197.0
2018-04   305B    PUNGGOL RD                             113      4227.2
2018-04   208A    PUNGGOL PL                             113      3518.1
2018-04   208A    PUNGGOL PL                             113      3518.1
2018-04   22      SIN MING RD                            113       644.2
2018-04   23      SIN MING RD                            113       288.2
2018-04   22      SIN MING RD                            113       644.2
2018-04   181B    BOON LAY DR                            113       628.4
2018-04   121     YUAN CHING RD                          113       298.1
2018-04   119     HO CHING RD                            113       353.7


### [ FULL DATASET ] — Compute distances for all rows
Rows are processed by month group using vectorised NumPy — typically completes in a few seconds.

In [12]:
print("Computing nearest open hawker distance for all rows (grouped by month)...")
dist_col = np.full(len(df), np.nan)

for month, group_idx in df.groupby('month').groups.items():
    coords            = open_hawkers_by_month.get(month, np.zeros((0, 2)))
    lats              = df.loc[group_idx, 'lat'].values.astype(float)
    lons              = df.loc[group_idx, 'lon'].values.astype(float)
    dist_col[group_idx] = hawker_dists_vectorized(lats, lons, coords)

df['dist_nearest_hawker_m'] = dist_col
non_null = int((~np.isnan(dist_col)).sum())
print(f"Done. Non-null values: {non_null:,} / {len(df):,}")

Computing nearest open hawker distance for all rows (grouped by month)...
Done. Non-null values: 197,327 / 197,432


## Phase 4 — Save Full Dataset

Print a summary of the nearest hawker distance column — null counts, statistics, and a town-level sanity check. No file is written here; all phases accumulate columns in memory and the dataset is saved once in the **Final Save** cell after Phase 7.

In [13]:
has_dist = df['dist_nearest_hawker_m'].notna().sum()
missing  = df['dist_nearest_hawker_m'].isna().sum()

print(f"\n{'='*60}")
print("SUMMARY — dist_nearest_hawker_m")
print(f"{'='*60}")
print(f"Non-null: {has_dist:,}   Null (failed geocode): {missing:,}")
print(f"  Min:  {df['dist_nearest_hawker_m'].min():.1f} m")
print(f"  Mean: {df['dist_nearest_hawker_m'].mean():.1f} m")
print(f"  Max:  {df['dist_nearest_hawker_m'].max():.1f} m")

town_hk = (df.groupby('town')['dist_nearest_hawker_m']
             .mean().sort_values()
             .rename('mean_dist_hawker_m').round(0).astype(int))
print(f"\nTop 5 towns closest to a hawker centre:")
print(town_hk.head(5).to_string())
print(f"\nTop 5 towns furthest from a hawker centre:")
print(town_hk.tail(5).to_string())


SUMMARY — dist_nearest_hawker_m
Non-null: 197,327   Null (failed geocode): 105
  Min:  0.0 m
  Mean: 1037.4 m
  Max:  4889.0 m

Top 5 towns closest to a hawker centre:
town
CENTRAL AREA     197
BUKIT MERAH      304
MARINE PARADE    314
ANG MO KIO       316
GEYLANG          337

Top 5 towns furthest from a hawker centre:
town
BUKIT TIMAH      1574
BUKIT BATOK      1594
SENGKANG         1674
PUNGGOL          2605
CHOA CHU KANG    2739


## Phase 5 — CBD Distance

Compute the straight-line distance from each flat to the Central Business District (CBD), using Raffles Place MRT as the proxy — consistent with Singapore hedonic pricing literature. No API calls are needed; the computation reuses `hawker_dists_vectorized()` with CBD as the sole target point.

In [14]:
# Raffles Place MRT used as CBD proxy, consistent with Singapore hedonic pricing literature
CBD_LAT = 1.2830
CBD_LON = 103.8513

# Reuse hawker_dists_vectorized with CBD as the sole target point.
# Returns NaN for rows with null lat/lon — consistent with dist_nearest_hawker_m behaviour.
cbd_coords = np.array([[CBD_LAT, CBD_LON]])
df['dist_cbd_m'] = hawker_dists_vectorized(
    df['lat'].values.astype(float),
    df['lon'].values.astype(float),
    cbd_coords
)

# ── Summary ──────────────────────────────────────────────────────────────────
non_null = df['dist_cbd_m'].notna().sum()
null_cnt = df['dist_cbd_m'].isna().sum()
print(f"\n{'='*60}")
print("SUMMARY — dist_cbd_m")
print(f"{'='*60}")
print(f"Non-null: {non_null:,}   Null (failed geocode): {null_cnt:,}")
print(f"  Min:  {df['dist_cbd_m'].min():,.1f} m")
print(f"  Mean: {df['dist_cbd_m'].mean():,.1f} m")
print(f"  Max:  {df['dist_cbd_m'].max():,.1f} m")

# ── Sanity check: mean CBD distance by town ──────────────────────────────────
town_cbd = (df.groupby('town')['dist_cbd_m']
              .mean()
              .sort_values()
              .rename('mean_dist_cbd_m')
              .round(0)
              .astype(int))
print(f"\nTop 5 closest towns to CBD:")
print(town_cbd.head(5).to_string())
print(f"\nTop 5 furthest towns from CBD:")
print(town_cbd.tail(5).to_string())


SUMMARY — dist_cbd_m
Non-null: 197,327   Null (failed geocode): 105
  Min:  591.8 m
  Mean: 12,539.6 m
  Max:  20,312.7 m

Top 5 closest towns to CBD:
town
CENTRAL AREA       1696
BUKIT MERAH        3387
KALLANG/WHAMPOA    4325
TOA PAYOH          6028
GEYLANG            6171

Top 5 furthest towns from CBD:
town
YISHUN           15981
CHOA CHU KANG    16365
JURONG WEST      17261
WOODLANDS        18548
SEMBAWANG        18821


## Phase 6 — Primary School Distance

Geocode all MOE primary schools by postal code using the OneMap search API, then compute the straight-line distance from each HDB flat to the nearest primary school. School coordinates are cached in `data/school_geocode_cache.json` — re-running this cell skips schools that are already cached.

In [15]:
import requests
import time
from dotenv import load_dotenv

# ── Load credentials (needed for OneMap search API) ──────────────────────────
load_dotenv('../.env')
ONEMAP_EMAIL    = os.getenv('ONEMAP_EMAIL')
ONEMAP_PASSWORD = os.getenv('ONEMAP_PASSWORD')

_token   = {"access_token": None, "expiry": 0}
_session = requests.Session()

def _get_token():
    if _token["access_token"] and time.time() < int(_token["expiry"]) - 60:
        return _token["access_token"]
    resp = _session.post(
        "https://www.onemap.gov.sg/api/auth/post/getToken",
        json={"email": ONEMAP_EMAIL, "password": ONEMAP_PASSWORD}, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    _token["access_token"] = data["access_token"]
    _token["expiry"] = int(data["expiry_timestamp"])
    return _token["access_token"]

# ── Load and filter MOE school directory ─────────────────────────────────────
SCHOOLS_CSV = '../data/Generalinformationofschools.csv'
schools_raw = pd.read_csv(SCHOOLS_CSV)
schools = (schools_raw[schools_raw['mainlevel_code'] == 'PRIMARY']
             [['school_name', 'address', 'postal_code']]
             .copy()
             .reset_index(drop=True))
print(f"Primary schools loaded: {len(schools)}")

# ── Geocode by postal code ────────────────────────────────────────────────────
SCHOOL_GEOCODE_CACHE = '../data/school_geocode_cache.json'
if os.path.exists(SCHOOL_GEOCODE_CACHE):
    with open(SCHOOL_GEOCODE_CACHE) as f:
        school_cache = json.load(f)
    print(f"Loaded existing cache: {len(school_cache)} postal codes")
else:
    school_cache = {}
    print("No existing cache — starting fresh.")

GEOCODE_URL = 'https://www.onemap.gov.sg/api/common/elastic/search'
new_calls   = 0

for i, row in schools.iterrows():
    postal = str(row['postal_code']).zfill(6)
    if postal in school_cache:
        continue                             # already cached — skip API call
    try:
        resp = _session.get(
            GEOCODE_URL,
            params={'searchVal': postal, 'returnGeom': 'Y',
                    'getAddrDetails': 'Y', 'pageNum': 1},
            headers={'Authorization': f'Bearer {_get_token()}'},
            timeout=10)
        resp.raise_for_status()
        results = resp.json().get('results', [])
        if results:
            school_cache[postal] = {
                'lat': float(results[0]['LATITUDE']),
                'lon': float(results[0]['LONGITUDE']),
            }
        else:
            print(f"  WARNING: no results for {row['school_name']} (postal {postal})")
            school_cache[postal] = None
    except requests.RequestException as e:
        print(f"  ERROR geocoding {row['school_name']}: {e}")
        school_cache[postal] = None
    new_calls += 1
    time.sleep(0.3)
    if new_calls % 10 == 0:                 # checkpoint every 10 new calls
        with open(SCHOOL_GEOCODE_CACHE, 'w') as f:
            json.dump(school_cache, f)
        print(f"  Checkpoint saved ({new_calls} new API calls so far...)")

# Final save
with open(SCHOOL_GEOCODE_CACHE, 'w') as f:
    json.dump(school_cache, f)

ok  = sum(1 for v in school_cache.values() if v is not None)
bad = sum(1 for v in school_cache.values() if v is None)
print(f"\nGeocoding complete: {ok} succeeded, {bad} failed")
print(f"Cache saved to: {SCHOOL_GEOCODE_CACHE}")

Primary schools loaded: 179
Loaded existing cache: 179 postal codes

Geocoding complete: 179 succeeded, 0 failed
Cache saved to: ../data/school_geocode_cache.json


In [16]:
# Build coordinate array for successfully geocoded primary schools
school_coords = np.array([
    [v['lat'], v['lon']]
    for v in school_cache.values()
    if v is not None
], dtype=float)
print(f"Using {len(school_coords)} geocoded primary schools for distance computation.")

# Vectorized minimum-distance computation — reuses hawker_dists_vectorized()
df['dist_nearest_primary_m'] = hawker_dists_vectorized(
    df['lat'].values.astype(float),
    df['lon'].values.astype(float),
    school_coords
)

# ── Summary ──────────────────────────────────────────────────────────────────
non_null = df['dist_nearest_primary_m'].notna().sum()
null_cnt = df['dist_nearest_primary_m'].isna().sum()
print(f"\n{'='*60}")
print("SUMMARY — dist_nearest_primary_m")
print(f"{'='*60}")
print(f"Non-null: {non_null:,}   Null (failed geocode): {null_cnt:,}")
print(f"  Min:  {df['dist_nearest_primary_m'].min():,.1f} m")
print(f"  Mean: {df['dist_nearest_primary_m'].mean():,.1f} m")
print(f"  Max:  {df['dist_nearest_primary_m'].max():,.1f} m")

# ── Sanity check: mean primary school distance by town ───────────────────────
town_sch = (df.groupby('town')['dist_nearest_primary_m']
              .mean()
              .sort_values()
              .rename('mean_dist_primary_m')
              .round(0)
              .astype(int))
print(f"\nTop 5 towns with closest primary schools:")
print(town_sch.head(5).to_string())
print(f"\nTop 5 towns with furthest primary schools:")
print(town_sch.tail(5).to_string())

Using 179 geocoded primary schools for distance computation.

SUMMARY — dist_nearest_primary_m
Non-null: 197,327   Null (failed geocode): 105
  Min:  43.8 m
  Mean: 421.0 m
  Max:  3,296.6 m

Top 5 towns with closest primary schools:
town
MARINE PARADE    273
PUNGGOL          284
SENGKANG         306
BUKIT PANJANG    346
BUKIT TIMAH      349

Top 5 towns with furthest primary schools:
town
CLEMENTI           604
QUEENSTOWN         609
KALLANG/WHAMPOA    609
BISHAN             689
JURONG EAST        701


## Phase 7 — Parks Distance

Load NParks managed green spaces from `data/Parks.geojson`, exclude non-park features (playgrounds, car parks, terminals, etc.), then compute the straight-line distance from each HDB flat to the nearest retained park. No API calls needed — all computation is local.

In [17]:
PARKS_GEOJSON = '../data/Parks.geojson'

with open(PARKS_GEOJSON) as f:
    parks_geojson = json.load(f)

rows = []
for feat in parks_geojson['features']:
    props = feat['properties']
    lon, lat = feat['geometry']['coordinates']   # GeoJSON standard: [lon, lat]
    rows.append({'name': props.get('NAME', ''), 'lat': lat, 'lon': lon})
parks_df = pd.DataFrame(rows)
print(f"Total features loaded: {len(parks_df)}")

# Exclusion rules — applied to uppercase NAME field
name = parks_df['name']
rules = {
    'ends with PLAYGROUND':    name.str.endswith('PLAYGROUND',    na=False),
    'ends with TERMINAL':      name.str.endswith('TERMINAL',      na=False),
    'ends with NURSERY':       name.str.endswith('NURSERY',       na=False),
    'ends with STATELAND':     name.str.endswith('STATELAND',     na=False),
    'ends with LINKWAY':       name.str.endswith('LINKWAY',       na=False),
    'contains CAR PARK':       name.str.contains('CAR PARK',      na=False),
    'contains FOOTBALL FIELD': name.str.contains('FOOTBALL FIELD', na=False),
    'contains SPORTSG':        name.str.contains('SPORTSG',       na=False),
}
for label, mask in rules.items():
    print(f"  Excluded ({label}): {mask.sum()}")

excl_mask = pd.concat(rules.values(), axis=1).any(axis=1)
parks_df  = parks_df[~excl_mask].reset_index(drop=True)
print(f"Parks after filtering: {len(parks_df)}")

Total features loaded: 450
  Excluded (ends with PLAYGROUND): 24
  Excluded (ends with TERMINAL): 1
  Excluded (ends with NURSERY): 1
  Excluded (ends with STATELAND): 1
  Excluded (ends with LINKWAY): 1
  Excluded (contains CAR PARK): 1
  Excluded (contains FOOTBALL FIELD): 1
  Excluded (contains SPORTSG): 1
Parks after filtering: 419


In [18]:
park_coords = parks_df[['lat', 'lon']].values.astype(float)
print(f"Using {len(park_coords)} parks for distance computation.")

# Vectorized minimum-distance computation — reuses hawker_dists_vectorized()
df['dist_nearest_park_m'] = hawker_dists_vectorized(
    df['lat'].values.astype(float),
    df['lon'].values.astype(float),
    park_coords
)

non_null = df['dist_nearest_park_m'].notna().sum()
null_cnt = df['dist_nearest_park_m'].isna().sum()
print(f"\n{'='*60}")
print("SUMMARY — dist_nearest_park_m")
print(f"{'='*60}")
print(f"Non-null: {non_null:,}   Null (failed geocode): {null_cnt:,}")
print(f"  Min:  {df['dist_nearest_park_m'].min():,.1f} m")
print(f"  Mean: {df['dist_nearest_park_m'].mean():,.1f} m")
print(f"  Max:  {df['dist_nearest_park_m'].max():,.1f} m")

town_pk = (df.groupby('town')['dist_nearest_park_m']
             .mean().sort_values()
             .rename('mean_dist_park_m').round(0).astype(int))
print(f"\nTop 5 towns closest to a park:")
print(town_pk.head(5).to_string())
print(f"\nTop 5 towns furthest from a park:")
print(town_pk.tail(5).to_string())

Using 419 parks for distance computation.

SUMMARY — dist_nearest_park_m
Non-null: 197,327   Null (failed geocode): 105
  Min:  44.3 m
  Mean: 759.8 m
  Max:  2,217.4 m

Top 5 towns closest to a park:
town
CENTRAL AREA     299
BUKIT TIMAH      374
MARINE PARADE    392
BISHAN           440
GEYLANG          448

Top 5 towns furthest from a park:
town
PUNGGOL           956
BUKIT PANJANG    1033
JURONG EAST      1155
BUKIT BATOK      1177
WOODLANDS        1218


## Phase 8 — SportSG Sport Facility Distance

Load SportSG managed sport facilities from `data/SportSGSportFacilitiesGEOJSON.geojson` and compute the straight-line distance from each HDB flat to the nearest facility. No exclusion filtering is needed — all 45 entries are legitimate sport facilities. No API calls required.

In [19]:
SPORTSG_GEOJSON = '../data/SportSGSportFacilitiesGEOJSON.geojson'

with open(SPORTSG_GEOJSON) as f:
    sportsg_geojson = json.load(f)

rows = []
for feat in sportsg_geojson['features']:
    props = feat['properties']
    lon, lat = feat['geometry']['coordinates']   # GeoJSON standard: [lon, lat]
    rows.append({'venue': props.get('VENUE', ''), 'lat': lat, 'lon': lon})
sportsg_df = pd.DataFrame(rows)

print(f"Total SportSG facilities loaded: {len(sportsg_df)}")
print("\nVenue list:")
for _, row in sportsg_df.iterrows():
    print(f"  {row['venue']}")

# Vectorized minimum-distance computation — reuses hawker_dists_vectorized()
sportsg_coords = sportsg_df[['lat', 'lon']].values.astype(float)
df['dist_nearest_sportsg_m'] = hawker_dists_vectorized(
    df['lat'].values.astype(float),
    df['lon'].values.astype(float),
    sportsg_coords
)

non_null = df['dist_nearest_sportsg_m'].notna().sum()
null_cnt = df['dist_nearest_sportsg_m'].isna().sum()
print(f"\n{'='*60}")
print("SUMMARY — dist_nearest_sportsg_m")
print(f"{'='*60}")
print(f"Non-null: {non_null:,}   Null (failed geocode): {null_cnt:,}")
print(f"  Min:  {df['dist_nearest_sportsg_m'].min():,.1f} m")
print(f"  Mean: {df['dist_nearest_sportsg_m'].mean():,.1f} m")
print(f"  Max:  {df['dist_nearest_sportsg_m'].max():,.1f} m")

town_sg = (df.groupby('town')['dist_nearest_sportsg_m']
             .mean().sort_values()
             .rename('mean_dist_sportsg_m').round(0).astype(int))
print(f"\nTop 5 towns closest to a SportSG facility:")
print(town_sg.head(5).to_string())
print(f"\nTop 5 towns furthest from a SportSG facility:")
print(town_sg.tail(5).to_string())

Total SportSG facilities loaded: 45

Venue list:
  Bukit Canberra Sport Centre
  Bukit Gombak Sport Centre
  Burghley Squash & Tennis Centre
  Choa Chu Kang Sport Centre
  Clementi Sport Centre
  Clementi Stadium
  Delta Sport Centre
  Geylang East Swimming Complex
  Heartbeat@Bedok ActiveSG Sport Centre
  Hougang Sport Centre
  Jalan Besar Sport Centre
  Jurong East Sport Centre
  Jurong Lake Gardens ActiveSG Gym & Pool
  Jurong West Sport Centre
  Kallang Basin Swimming Complex
  ActiveSG Sport Park @ Bedok North
  ActiveSG Gym@Ang Mo Kio Community Centre
  ActiveSG Gym@Enabling Village
  ActiveSG Gym@Fernvale Square
  ActiveSG Gym@Serangoon Central
  ActiveSG Gym@Toa Payoh
  ActiveSG Gym@Toa Payoh West Community Centre
  ActiveSG Hockey Village @ Boon Lay
  ActiveSG Sport Village@Jurong Town
  Ang Mo Kio Swimming Complex
  Bedok Stadium
  Bishan Sport Centre
  Bishan Clubhouse
  Bukit Batok Swimming Complex
  Kallang Sport Centre
  Katong Swimming Complex
  MOE (Evans)
  Tampines Sp

## Phase 9 — Shopping Mall Distance

Load shopping malls from `data/shoppingmalls.csv` (238 rows). Deduplicate by mall name — multi-entrance malls and data-entry duplicates are collapsed by taking the mean lat/lon per name group, leaving 221 unique mall locations. Compute straight-line distance from each HDB flat to the nearest mall using `hawker_dists_vectorized()`. No API calls required.

In [22]:
MALLS_CSV = '../data/shoppingmalls.csv'
malls_raw = pd.read_csv(MALLS_CSV)
print(f"Total rows loaded: {len(malls_raw)}")

# Deduplicate by name — multi-entrance malls and data-entry duplicates
malls_df = (malls_raw.groupby('name', as_index=False)
              .agg(lat=('lat', 'mean'), lon=('lon', 'mean')))
n_dupes = len(malls_raw) - len(malls_df)
print(f"Unique malls after deduplication: {len(malls_df)}")
print(f"Duplicate rows collapsed: {n_dupes}")

# Vectorized minimum-distance computation — reuses hawker_dists_vectorized()
mall_coords = malls_df[['lat', 'lon']].values.astype(float)
df['dist_nearest_mall_m'] = hawker_dists_vectorized(
    df['lat'].values.astype(float),
    df['lon'].values.astype(float),
    mall_coords
)

non_null = df['dist_nearest_mall_m'].notna().sum()
null_cnt = df['dist_nearest_mall_m'].isna().sum()
print(f"\n{'='*60}")
print("SUMMARY — dist_nearest_mall_m")
print(f"{'='*60}")
print(f"Non-null: {non_null:,}   Null (failed geocode): {null_cnt:,}")
print(f"  Min:  {df['dist_nearest_mall_m'].min():,.1f} m")
print(f"  Mean: {df['dist_nearest_mall_m'].mean():,.1f} m")
print(f"  Max:  {df['dist_nearest_mall_m'].max():,.1f} m")

town_mall = (df.groupby('town')['dist_nearest_mall_m']
               .mean().sort_values()
               .rename('mean_dist_mall_m').round(0).astype(int))
print(f"\nTop 5 towns closest to a shopping mall:")
print(town_mall.head(5).to_string())
print(f"\nTop 5 towns furthest from a shopping mall:")
print(town_mall.tail(5).to_string())

Total rows loaded: 238
Unique malls after deduplication: 221
Duplicate rows collapsed: 17

SUMMARY — dist_nearest_mall_m
Non-null: 197,327   Null (failed geocode): 105
  Min:  5.9 m
  Mean: 642.7 m
  Max:  2,961.7 m

Top 5 towns closest to a shopping mall:
town
CENTRAL AREA     319
CHOA CHU KANG    422
SENGKANG         472
BUKIT PANJANG    477
BUKIT TIMAH      510

Top 5 towns furthest from a shopping mall:
town
GEYLANG             787
KALLANG/WHAMPOA     822
MARINE PARADE       848
JURONG EAST        1017
BEDOK              1158


In [24]:
# ── Final Save — Write Fully Enriched Dataset ────────────────────────────────
# Columns saved:
# dist_nearest_hawker_m   — nearest open hawker centre at transaction month
# dist_cbd_m              — straight-line distance to Raffles Place MRT (CBD proxy)
# dist_nearest_primary_m  — nearest MOE primary school
# dist_nearest_park_m     — nearest NParks managed green space
# dist_nearest_sportsg_m  — nearest SportSG managed sport facility
# dist_nearest_mall_m     — nearest shopping mall (221 malls, deduplicated)

OUTPUT_FILE = '../merged_data/hdb_with_amenities_macro.csv'
df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(df):,} rows to: {OUTPUT_FILE}\n")

dist_cols = [
    'dist_nearest_hawker_m',
    'dist_cbd_m',
    'dist_nearest_primary_m',
    'dist_nearest_park_m',
    'dist_nearest_sportsg_m',
    'dist_nearest_mall_m',
]
summary = pd.DataFrame({
    'non_null': [df[c].notna().sum() for c in dist_cols],
    'null':     [df[c].isna().sum()  for c in dist_cols],
    'min_m':    [round(df[c].min(), 1) for c in dist_cols],
    'mean_m':   [round(df[c].mean(), 1) for c in dist_cols],
    'max_m':    [round(df[c].max(), 1) for c in dist_cols],
}, index=dist_cols)
print(summary.to_string())

Saved 197,432 rows to: ../merged_data/hdb_with_amenities_macro.csv

                        non_null  null  min_m   mean_m    max_m
dist_nearest_hawker_m     197327   105    0.0   1037.4   4889.0
dist_cbd_m                197327   105  591.8  12539.6  20312.7
dist_nearest_primary_m    197327   105   43.8    421.0   3296.6
dist_nearest_park_m       197327   105   44.3    759.8   2217.4
dist_nearest_sportsg_m    197327   105   42.6   1140.8   4296.1
dist_nearest_mall_m       197327   105    5.9    642.7   2961.7
